# Functional evaluation of LLM outputs

We will use several strategies to evaluate if the LLM produces a 'correct' WDL workflow, not merely one that will run. We want to know if the WDL will perform the tasks the user has requested.

We will employ reference-based evaluation using pre-built WILDS WDL pipelines as our ground truth. We will have five prompt-ground truth pairs, using these pipelines:
- bwa-gatk (basic)
- sra-star (basic)
- sra-salmon (basic)
- star-deseq2 (intermediate)
- saturation (intermediate)

Other thoughts:
- Don't have the model output meta/parameter meta sections because that makes eval harder and that stuff isn't functional anyway

Separately, we should confirm that WDL tasks being retrieved from our RAG database will indeed meet the users request. We expect that they always will because we will use keyword filtering based on user input terms but we should still confirm this (and confirm we can choose between equivalent tasks).

## Imports

In [10]:
# Lexical similarity imports
from rapidfuzz.distance import Levenshtein

# Semantic similarity imports
from llama_index.embeddings.huggingface import HuggingFaceEmbedding
import numpy as np

# Rule-based check imports
from pathlib import Path
import re

## Loading Reference WDL as Text

Read the 'ground truth' wdl for a given benchmarking case. I have included a slightly off WDL that still passes `sprocket lint` and `sprocket check`.

In [40]:
# Customize for your system
GROUND_TRUTH_DIR = 'evals/data'

def load_ground_truth(filename: str) -> str:
    """Return a reference WDL as a string."""
    full_wdl = Path(GROUND_TRUTH_DIR, filename).read_text()
    return full_wdl

In [41]:
test_truth_wdl = load_ground_truth('ww-sra-star.wdl')
test_false_wdl = load_ground_truth('ww-sra-star_wrong.wdl')

# Show first 400 characters of each wdl
print(test_truth_wdl[:400])
print('\n-----------------------\n')
print(test_false_wdl[:400])

version 1.0

import "https://raw.githubusercontent.com/getwilds/wilds-wdl-library/refs/heads/main/modules/ww-sra/ww-sra.wdl" as sra_tasks
import "https://raw.githubusercontent.com/getwilds/wilds-wdl-library/refs/heads/main/modules/ww-star/ww-star.wdl" as star_tasks

struct RefGenome {
    String name
    File fasta
    File gtf
}

workflow sra_star {
  input {
    Array[String] sra_id_list
    Ref

-----------------------

version 1.0

import "https://raw.githubusercontent.com/getwilds/wilds-wdl-library/refs/heads/main/modules/ww-sra/ww-sra.wdl" as sra_tasks
import "https://raw.githubusercontent.com/getwilds/wilds-wdl-library/refs/heads/main/modules/ww-star/ww-star.wdl" as star_tasks

struct MYgenome {
    String name
    File fasta
    File gtf
}

workflow sra_star {
  input {
    Array[String] sra_id_list
    MYge


## Lexical (text) similarity

Use the simple Levenshtein distance because code-specific textual similarity metrics rely on proper parsing of WDL and there isn't a lot of tooling for WDL syntax yet.

In [42]:
def lexical_similarity(ref: str, to_eval: str) -> float:
    """Measure text similarity"""
    return 1 - Levenshtein.normalized_distance(ref, to_eval)

Let's say it has to have a score of >= 0.85 to pass.

In [ ]:
pass_score_lexical = 0.85

# Toy example
truth="The Eiffel Tower is located in Paris."
generated="The Eiffel Tower is located in India."

lexical_score = lexical_similarity(ref=truth, to_eval=generated)

print(lexical_score)
print(lexical_score >= pass_score_lexical)

0.8918918918918919
True


Let's try comparing WDLs

In [43]:
lexical_score = lexical_similarity(ref=test_truth_wdl, to_eval=test_false_wdl)

print(lexical_score)
print(lexical_score >= pass_score_lexical)

0.8722532588454376
True


## Semantic (meaning) similarity

Measure cosine similarity of embeddings between ground truth and generated WDL. I originally was going to use codeBERT, which, while  not trained on WDL (or Rust, or many other langauges for that matter), should be able to 'understand' the syntax enough to work. However, it's fairly large and so I decided it was more pragmatic to use the much smaller `all-MiniLM-L6-v2` model. This is also the ChromaDB default and therefore the model used for embeddings in our ChromaDB document store.

In [51]:
embed_model = HuggingFaceEmbedding(model_name="sentence-transformers/all-MiniLM-L6-v2")

def semantic_similarity(embed_model: HuggingFaceEmbedding, ref: str, to_eval: str) -> float:
    """Measure 'meaning' similarity by comparing text embeddings."""
    emb_ref = np.array(embed_model.get_text_embedding(ref))
    emb_eval = np.array(embed_model.get_text_embedding(to_eval))
    # Calculate cosine similarity with numpy
    dot_product = np.dot(emb_ref, emb_eval)
    product_of_lengths = np.linalg.norm(emb_ref) * np.linalg.norm(emb_eval)
    return float(dot_product / product_of_lengths)

Loading weights: 100%|██████████| 103/103 [00:00<00:00, 10683.09it/s]


Let's say it has to have a score of >= 0.90 to pass

In [52]:
pass_score_semantic = 0.9

# Toy example
truth="The Eiffel Tower is located in Paris."
generated="The Eiffel Tower is located in India."

semantic_score = semantic_similarity(embed_model=embed_model, ref=truth, to_eval=generated)

print(semantic_score)
print(semantic_score >= pass_score_semantic)

0.8239411917563605
False


Let's try comparing WDLs

In [53]:
semantic_score = semantic_similarity(embed_model=embed_model, ref=test_truth_wdl, to_eval=test_false_wdl)

print(semantic_score)
print(semantic_score >= pass_score_semantic)

0.994064218990574
True


## Rule-based metric

Ensure that all retrieved WDL tasks and only those retrieved WDL tasks are used in the final output.

Note: This assuemes that the retrieval was done correctly (that needs to be evaluated separately)

In [ ]:
def check_retrieved_module_usage(case: dict, to_eval: str) -> bool:
    """Confirm that retrieved modules appear in the output WDL imports."""
    expected = {m.strip() for m in case["modules"].split(",")}
    found = set(re.findall(r'ww-[\w-]+(?=/ww-[\w-]+\.wdl)', to_eval))
    return expected == found

In [58]:
test_case =   {
    "id": "basic_sra_star",
    "pipeline_name": "sra-star",
    "workflow_name": "sra_star",
    "modules": "ww-sra, ww-star",
    "analysis_goal": "Download SRA accessions and align reads with STAR two-pass",
    "input_data_type": "SRA accession IDs",
    "organism": "Homo sapiens",
    "reference_genome": "GRCh38",
    "ground_truth_file": "ww-sra-star.wdl"
  }


check_retrieved_module_usage(case=test_case, to_eval=test_false_wdl)

True